# CHANCE-C Quickstarter Guide

Welcome to the **CHANCE-C** quickstart guide! 

This notebook will walk you through:
- Setting up and configuring CHANCE-C
- Understanding input data requirements
- Using the field mapping system for custom data
- Running your first simulation
- Visualizing and analyzing results

Let's get started!


## What is CHANCE-C?

CHANCE-C is a powerful agent-based modeling (ABM) framework designed for simulating urban housing markets, population dynamics, and weather risk interactions at the community scale.

### Key Features:
- **Agent-Based Modeling**: Simulates individual households and their housing decisions
- **Extreme Weather Risk Integration**: Models flood risk and weather adaptation behaviors
- **Flexible Data Input**: Supports custom data formats through field mapping
- **Spatial Analysis**: Works with geographic data (shapefiles, GeoJSON)
- **Policy Simulation**: Test interventions like zoning changes, flood insurance policies
- **Visualization Tools**: Built-in plotting and analysis capabilities

### Use Cases:
- Urban planning and policy analysis
- Extreme weather adaptation research
- Housing market dynamics studies


## Required Input Data

CHANCE-C requires five types of input files:

1. **Geographic File** (`geo_filename`): Shapefile with census block groups
   - Contains geometry and geographic identifiers

2. **Population File** (`pop_filename`): CSV with population data
   - Population counts by block group

3. **Flood File** (`flood_filename`): CSV with flood risk data
   - Flood zone percentages and areas

4. **Housing File** (`housing_filename`): CSV with housing characteristics
   - Historical housing prices, income, household size data

5. **Hedonic File** (`hedonic_filename`): CSV with hedonic regression results
   - Normalized housing characteristics and residuals

### Field Mapping System

CHANCE-C includes a flexible **field mapping system** that allows you to use data files with different column names. Instead of renaming your columns, you can create a mapping file that tells CHANCE-C how to interpret your data.

**For detailed information on field mapping, see**: [`chance_c/data/FIELD_MAPPING_README.md`](../chance_c/data/FIELD_MAPPING_README.md)


## Step 1: Import CHANCE-C

Let's start by importing the main Model class and other necessary modules:


In [1]:
import os

# Import CHANCE-C
from chance_c import Model
from chance_c.data_loader import SimulationConfig

print("CHANCE-C imported successfully!")
print(f"Current working directory: {os.getcwd()}")


CHANCE-C imported successfully!
Current working directory: /Users/d3y010/repos/github/icom_abm/notebooks


## Step 2: Set Up Configuration

CHANCE-C uses a configuration system to manage simulation parameters and file paths. We'll use the example configuration file as a starting point.

### Option A: Use Example Configuration (Recommended for first-time users)


In [2]:
# Load the example configuration
config_path = os.path.join('chance_c', 'data', 'example_config.yml')

try:
    config = SimulationConfig.from_yaml(config_path)
    print("Configuration loaded successfully!")
    print(f"Simulation: {config.simulation_name}")
    print(f"Scenario: {config.scenario}")
    print(f"Years: {config.start_year} to {config.start_year + config.n_years}")
    print(f"Agent aggregation: {config.agent_housing_aggregation}")
    
except FileNotFoundError:
    print("Example config file not found. Creating a default configuration...")
    config = SimulationConfig()
    print("Default configuration created!")


Example config file not found. Creating a default configuration...
Default configuration created!


### Option B: Create Custom Configuration

If you have your own data files, you can create a custom configuration:


In [ ]:
# Example: Create custom configuration for your own data
# Uncomment and modify the paths below to use your own data files

# custom_config = SimulationConfig(
#     simulation_name="My_Custom_Simulation",
#     scenario="Custom_Scenario",
#     start_year=2020,
#     n_years=5,
#     
#     # File paths (update these to point to your data files)
#     geo_filename="path/to/your/geography.shp",
#     pop_filename="path/to/your/population.csv",
#     flood_filename="path/to/your/flood_data.csv",
#     housing_filename="path/to/your/housing_data.csv",
#     hedonic_filename="path/to/your/hedonic_data.csv",
#     
#     # Optional: Field mapping file if your columns have different names
#     field_mapping_file="path/to/your/field_mapping.yml"
# )

print("Tip: Uncomment the code above to create a custom configuration for your data!")


## Step 3: Understand Required Data Fields

Let's explore what data fields are required for each input file type:


In [3]:
# Check required columns for each file type
file_types = ['geo', 'pop', 'flood', 'housing', 'hedonic']

for file_type in file_types:
    try:
        required_cols = config.get_required_columns(file_type)
        print(f"\n{file_type.upper()} FILE REQUIREMENTS:")
        print("-" * 40)
        for field, description in required_cols.items():
            print(f"  {field}: {description}")
    except Exception as e:
        print(f"Could not get requirements for {file_type}: {e}")



GEO FILE REQUIREMENTS:
----------------------------------------
  GISJOIN: Required field: GISJOIN
  GEOID: Required field: GEOID
  COUNTYFP: Required field: COUNTYFP
  TRACTCE: Required field: TRACTCE
  BLKGRPCE: Required field: BLKGRPCE
  ALAND: Required field: ALAND
  geometry: Required field: geometry

POP FILE REQUIREMENTS:
----------------------------------------
  GISJOIN: Required field: GISJOIN
  AJWME001: Required field: AJWME001

FLOOD FILE REQUIREMENTS:
----------------------------------------
  GISJOIN: Required field: GISJOIN
  Shape_Area: Required field: Shape_Area
  fld_area: Required field: fld_area
  perc_fld_area: Required field: perc_fld_area

HOUSING FILE REQUIREMENTS:
----------------------------------------
  GISJOIN: Required field: GISJOIN
  pop1990: Required field: pop1990
  mhi1990: Required field: mhi1990
  hhsize1990: Required field: hhsize1990
  coastdist: Required field: coastdist
  cbddist: Required field: cbddist
  hhtrans1993: Required field: hhtrans1

## Step 4: Set Up Data File Paths

For this quickstart, we'll set up some example file paths. **You'll need to update these to point to your actual data files.**


In [4]:
# Example data directory (update this to point to your data)
data_directory = "/path/to/your/data"  # Update this path!

# Example file paths - update these to match your data files
geo_filename = os.path.join(data_directory, "blck_grp_extract_prj.shp")
pop_filename = os.path.join(data_directory, "balt_bg_population_2018.csv")
flood_filename = os.path.join(data_directory, "bg_perc_100yr_flood.csv")
housing_filename = os.path.join(data_directory, "bg_housing_1993.csv")
hedonic_filename = os.path.join(data_directory, "simple_anova_hedonic_without_flood_bg0418.csv")

# Check if files exist (they won't for this example, but this shows you how to check)
files_to_check = {
    "Geographic": geo_filename,
    "Population": pop_filename,
    "Flood": flood_filename,
    "Housing": housing_filename,
    "Hedonic": hedonic_filename
}

print("FILE STATUS CHECK:")
print("-" * 30)
for file_type, filepath in files_to_check.items():
    exists = os.path.exists(filepath)
    status = "Found" if exists else "Not found"
    print(f"{file_type:12}: {status}")

print("\nUpdate the data_directory variable above to point to your actual data files!")


FILE STATUS CHECK:
------------------------------
Geographic  : Not found
Population  : Not found
Flood       : Not found
Housing     : Not found
Hedonic     : Not found

Update the data_directory variable above to point to your actual data files!


## Step 5: Create and Configure the Model

Now we'll create the CHANCE-C model with our configuration. **Note**: This step will fail if the data files don't exist, but it shows you the proper workflow.


In [ ]:
try:
    # Create the model with custom file paths
    model = Model(
        config=config,
        sensitivity_run=True,  # Enable for faster testing
        geo_filename=geo_filename,
        pop_filename=pop_filename,
        flood_filename=flood_filename,
        housing_filename=housing_filename,
        hedonic_filename=hedonic_filename
    )
    
    print("Model created successfully!")
    print(f"Landscape: {model.config.landscape_name}")
    print(f"Total block groups: {len(model.simulator.network.nodes)}")
    
except FileNotFoundError as e:
    print(f"Data file not found: {e}")
    print("To run the simulation, you need to:")
    print("   1. Update the data_directory path above")
    print("   2. Ensure all required data files exist")
    print("   3. Optionally create a field mapping file if your columns have different names")
    
except Exception as e:
    print(f"Error creating model: {e}")
    print("This might be due to missing data files or incorrect field names")


## Step 6: Run the Simulation

If the model was created successfully, we can run the simulation:


In [ ]:
try:
    # Check if model exists from previous step
    if 'model' in locals():
        print("Starting simulation...")
        print(f"Running for {model.config.n_years} years...")
        
        # Run the simulation
        model.run_simulation()
        
        print("Simulation completed successfully!")
        print(f"Final timestep: {model.simulator.network.current_timestep_idx}")
        
    else:
        print("Model not available. Please run the previous step successfully first.")
        
except Exception as e:
    print(f"Error running simulation: {e}")
    print("Make sure your data files are properly formatted and accessible")


## Step 7: Visualize Results

CHANCE-C provides several built-in visualization methods. Let's explore some of them:


In [ ]:
if 'model' in locals():
    try:
        print("Creating visualizations...")
        
        # Plot initial population distribution
        print("\n1. Initial Population Distribution")
        model.plot_initial_population()
        
        # Plot final population distribution
        print("\n2. Final Population Distribution")
        model.plot_final_population()
        
        # Plot population change
        print("\n3. Population Change Over Time")
        model.plot_population_change()
        
        # View network properties
        print("\n4. Network Properties")
        model.view_network_properties()
        
    except Exception as e:
        print(f"Error creating visualizations: {e}")
        print("Some plotting functions may require additional dependencies or specific data formats")
        
else:
    print("Model not available. Please run the simulation successfully first.")
    print("\nAvailable visualization methods when model is ready:")
    print("   - model.plot_initial_population()")
    print("   - model.plot_final_population()")
    print("   - model.plot_population_change()")
    print("   - model.plot_population_side_by_side()")
    print("   - model.plot_residuals_with_basemap()")
    print("   - model.view_network_properties()")


## Step 8: Analyze Results

Let's explore some analysis capabilities:


In [ ]:
if 'model' in locals():
    try:
        print("Analyzing simulation results...")
        
        # Get agent location history (example for agent ID 25000)
        print("\n1. Agent Location History")
        agent_history = model.get_agent_location_history(agent_id=25000)
        if agent_history:
            print(f"   Agent 25000 location history: {agent_history}")
        else:
            print("   No history found for agent 25000")
        
        # Combine housing dataframes for analysis
        print("\n2. Combined Housing Data")
        combined_df = model.combine_housing_dataframes()
        print(f"   Combined data shape: {combined_df.shape}")
        print(f"   Columns: {list(combined_df.columns)}")
        
        # Show some basic statistics
        print("\n3. Basic Statistics")
        if len(combined_df) > 0:
            print(f"   Average population: {combined_df['population'].mean():.1f}")
            print(f"   Population std dev: {combined_df['population'].std():.1f}")
            print(f"   Average new price: ${combined_df['new_price'].mean():.0f}")
        
    except Exception as e:
        print(f"Error analyzing results: {e}")
        
else:
    print("Model not available. Please run the simulation successfully first.")
    print("\nAvailable analysis methods when model is ready:")
    print("   - model.get_agent_location_history(agent_id)")
    print("   - model.combine_housing_dataframes()")
    print("   - Access to model.simulator.network for detailed analysis")


## Next Steps and Resources

Congratulations! You've completed the CHANCE-C quickstart guide. Here's what to do next:

### Immediate Next Steps:
1. **Get your data ready**: Prepare your geographic, population, flood, housing, and hedonic data files
2. **Set up field mapping**: If your column names differ from the required names, create a field mapping file
3. **Update file paths**: Modify the `data_directory` and file paths in this notebook to point to your data
4. **Run your first real simulation**: Execute the cells above with your actual data

### Learn More:
- **Field Mapping Guide**: [`chance_c/data/FIELD_MAPPING_README.md`](../chance_c/data/FIELD_MAPPING_README.md)
- **Example Configuration**: [`chance_c/data/example_config.yml`](../chance_c/data/example_config.yml)
- **Example Field Mapping**: [`chance_c/data/example_field_mapping.yml`](../chance_c/data/example_field_mapping.yml)

### Advanced Usage:
- Modify simulation parameters in your config file
- Create custom field mapping for your data
- Explore additional plotting and analysis methods
- Extend the model with custom engines or agents

### Tips for Success:
- Start with small datasets and short simulation periods for testing
- Use the field mapping system instead of renaming your data columns
- Check the console output for helpful error messages and debugging info
- Refer to the documentation and example files when in doubt

### Need Help?
- Check the error messages - they often contain helpful information
- Verify your data file formats and column names
- Ensure all required fields are present in your data
- Review the field mapping documentation for troubleshooting

Happy modeling with CHANCE-C!
